# Classification des Schémas d'Image
## Comparaison : Approche Symbolique (FrameNet) vs Approche Neuronale (BERT)

Ce notebook compare deux approches pour classifier les **schémas d'image** (Image Schemas) :
- **Approche symbolique** : prédictions basées sur des rôles FrameNet
- **Approche neuronale** : fine-tuning de `bert-base-uncased` via l'API HuggingFace Trainer

Les deux approches sont évaluées sur le **même jeu de données** avec les **mêmes métriques**.

---
## 1. Installation

In [ ]:
!pip install transformers datasets evaluate accelerate lime -q

---
## 2. Chargement des données

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, confusion_matrix, accuracy_score

RANDOM_SEED = 42

In [ ]:
# --- Dataset 1 : exemples annotés Image Schemas ---
!wget -q -O IS_repository.csv \
  "https://raw.githubusercontent.com/lwachowiak/Systematic-Analysis-of-Image-Schemas-through-Explainable-Multilingual-Language-Models/main/Data/Image%20Schemas%20English%20and%20German.csv"

df = pd.read_csv("IS_repository.csv")
df.rename(columns={'IMAGE_SCHEMA_ANNOTATION': 'label'}, inplace=True)
df['label'] = df['label'].str.strip()                        # supprime les espaces parasites
df.drop_duplicates(subset="LinguisticExamples", inplace=True)

# Suppression des classes peu représentées
CLASSES_TO_REMOVE = ["LINK", "OBJECT", "SUBSTANCE", "SPLITTING", "SUPPORT", "COVERING"]
df = df[~df['label'].isin(CLASSES_TO_REMOVE)].reset_index(drop=True)

print(f"Dataset : {len(df)} exemples")
print(df['label'].value_counts())

In [ ]:
# --- Dataset 2 : prédictions symboliques FrameNet Roles ---
df_eval = pd.read_csv("/content/100_for_eval_fnroles_out.csv")
df_eval['label_normalized'] = df_eval['label'].str.lower().str.strip().str.replace('-', '_')

print(f"Dataset FrameNet eval : {len(df_eval)} exemples")
print(df_eval['label_normalized'].value_counts())

---
## 3. Approche Symbolique — FrameNet Roles

In [ ]:
def normalize_first_pred(pred_str):
    if pd.isna(pred_str):
        return None
    return pred_str.split(',')[0].strip().lower().replace('-', '_')

def label_in_pred(pred_str, gold):
    if pd.isna(pred_str):
        return False
    return gold in [p.strip().lower().replace('-', '_') for p in str(pred_str).split(',')]

df_eval['pred_first']    = df_eval['pred'].apply(normalize_first_pred)
df_eval['exact_match']   = df_eval['pred_first'] == df_eval['label_normalized']
df_eval['partial_match'] = df_eval.apply(lambda r: label_in_pred(r['pred'], r['label_normalized']), axis=1)

print(f"Accuracy exacte   : {df_eval['exact_match'].mean():.3f}")
print(f"Accuracy partielle: {df_eval['partial_match'].mean():.3f}")

In [ ]:
valid = df_eval['pred_first'].notna()
print("=== Rapport — Approche Symbolique (FrameNet) ===")
print(classification_report(
    df_eval.loc[valid, 'label_normalized'],
    df_eval.loc[valid, 'pred_first'],
    zero_division=0
))

---
## 4. Approche Neuronale — BERT

### 4.1 Architecture

```
Texte brut
    │
    ▼
AutoTokenizer (bert-base-uncased)
    │
    ▼
BertModel — encodeur 12 couches, hidden_size=768
    │  token [CLS]
    ▼
Dropout(0.1)   ← intégré dans BertForSequenceClassification
    ▼
Linear(768 → NUM_LABELS)
    ▼
CrossEntropyLoss
```

On utilise `AutoModelForSequenceClassification` qui encapsule cette architecture et s'intègre directement avec le `Trainer` HuggingFace.

### 4.2 Préparation des labels et du dataset

In [ ]:
from datasets import Dataset

# Encodage des labels (entiers)
LABELS = sorted(df['label'].unique())
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for i, l in enumerate(LABELS)}
NUM_LABELS = len(LABELS)

print(f"{NUM_LABELS} classes : {LABELS}")

df['label_id'] = df['label'].map(label2id)

# Split train / test stratifié
train_df, test_df = train_test_split(
    df[['LinguisticExamples', 'label_id']],
    test_size=0.2,
    stratify=df['label_id'],
    random_state=RANDOM_SEED
)
print(f"Train : {len(train_df)} | Test : {len(test_df)}")

# Conversion en Dataset HuggingFace
train_dataset = Dataset.from_pandas(train_df.rename(columns={'LinguisticExamples': 'text', 'label_id': 'label'}))
test_dataset  = Dataset.from_pandas(test_df.rename(columns={'LinguisticExamples': 'text', 'label_id': 'label'}))

### 4.3 Tokenisation

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset  = test_dataset.map(tokenize, batched=True)

print("Colonnes disponibles :", train_dataset.column_names)

### 4.4 Métriques d'évaluation

In [ ]:
import evaluate

metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return {
        "accuracy" : accuracy_score(labels, predictions),
        "macro_f1" : f1_score(labels, predictions, average="macro",    zero_division=0),
        "weighted_f1": f1_score(labels, predictions, average="weighted", zero_division=0),
    }

### 4.5 Chargement du modèle

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
)

print(f"Modèle : bert-base-uncased → {NUM_LABELS} classes")

### 4.6 Entraînement avec le Trainer HuggingFace

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="bert_image_schema",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    seed=RANDOM_SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

### 4.7 Évaluation finale BERT

In [ ]:
# Métriques globales
results = trainer.evaluate()
print("=== Évaluation BERT — Test set ===")
for k, v in results.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

In [ ]:
# Rapport par classe + matrice de confusion
preds_output = trainer.predict(test_dataset)
y_pred = np.argmax(preds_output.predictions, axis=1)
y_true = preds_output.label_ids

print("=== Rapport de classification — BERT ===")
print(classification_report(y_true, y_pred, target_names=LABELS, zero_division=0))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=LABELS, yticklabels=LABELS, cmap='Blues')
plt.title('Matrice de confusion — BERT')
plt.xlabel('Prédiction'); plt.ylabel('Vérité terrain')
plt.tight_layout(); plt.show()

In [ ]:
# Courbes d'entraînement
log_history = pd.DataFrame(trainer.state.log_history)

train_logs = log_history.dropna(subset=['loss'])
eval_logs  = log_history.dropna(subset=['eval_loss'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(train_logs['epoch'], train_logs['loss'],      marker='o', label='Train Loss')
ax1.plot(eval_logs['epoch'],  eval_logs['eval_loss'],  marker='o', label='Val Loss')
ax1.set_title('Loss'); ax1.set_xlabel('Époque'); ax1.legend()

ax2.plot(eval_logs['epoch'], eval_logs['eval_weighted_f1'], marker='o', label='Weighted F1')
ax2.plot(eval_logs['epoch'], eval_logs['eval_macro_f1'],    marker='o', label='Macro F1')
ax2.plot(eval_logs['epoch'], eval_logs['eval_accuracy'],    marker='o', label='Accuracy')
ax2.set_title('Métriques'); ax2.set_xlabel('Époque'); ax2.legend()

plt.tight_layout(); plt.show()

---
## 5. Évaluation croisée sur le Dataset FrameNet

On évalue BERT sur le dataset FrameNet pour comparer avec l'approche symbolique **sur le même jeu de données**.

In [ ]:
from datasets import Dataset as HFDataset

# Alignement des labels FrameNet avec les classes BERT
label_to_id_lower = {l.lower().replace('-', '_'): i for l, i in label2id.items()}

df_eval['bert_label_id'] = df_eval['label_normalized'].map(label_to_id_lower)

# Rapport des labels non-mappés
df_unmapped = df_eval[df_eval['bert_label_id'].isna()]
if len(df_unmapped):
    print(f"{len(df_unmapped)} exemples non-mappés (classe absente du modèle) :")
    print(df_unmapped['label_normalized'].value_counts())

df_cross = df_eval.dropna(subset=['bert_label_id']).copy()
df_cross['bert_label_id'] = df_cross['bert_label_id'].astype(int)
print(f"\n{len(df_cross)} exemples utilisables sur {len(df_eval)} pour la comparaison.")

# Création du dataset HuggingFace pour l'évaluation croisée
cross_dataset = HFDataset.from_pandas(
    df_cross.rename(columns={'tweet_text': 'text', 'bert_label_id': 'label'})[['text', 'label']]
)
cross_dataset = cross_dataset.map(tokenize, batched=True)

In [ ]:
# Prédictions BERT sur le dataset FrameNet
cross_output = trainer.predict(cross_dataset)
y_pred_cross = np.argmax(cross_output.predictions, axis=1)
y_true_cross = cross_output.label_ids

bert_cross_acc     = accuracy_score(y_true_cross, y_pred_cross)
bert_cross_mac_f1  = f1_score(y_true_cross, y_pred_cross, average='macro',    zero_division=0)
bert_cross_wei_f1  = f1_score(y_true_cross, y_pred_cross, average='weighted', zero_division=0)

print("=== Évaluation BERT — Dataset FrameNet ===")
print(classification_report(y_true_cross, y_pred_cross, target_names=LABELS, zero_division=0))

---
## 6. Comparaison directe : Symbolique vs BERT

In [ ]:
# Métriques symboliques sur le même sous-ensemble
df_common  = df_cross.copy()
valid_sym  = df_common['pred_first'].notna()

sym_acc = df_common['exact_match'].mean()
sym_mac = f1_score(df_common.loc[valid_sym, 'label_normalized'],
                   df_common.loc[valid_sym, 'pred_first'], average='macro',    zero_division=0)
sym_wei = f1_score(df_common.loc[valid_sym, 'label_normalized'],
                   df_common.loc[valid_sym, 'pred_first'], average='weighted', zero_division=0)

print("="*60)
print(f" COMPARAISON — Dataset FrameNet (n={len(df_common)})")
print("="*60)
print(f"{'Métrique':<25} {'Symbolique':>12} {'BERT':>12}")
print("-"*60)
print(f"{'Accuracy (exacte)':<25} {sym_acc:>12.3f} {bert_cross_acc:>12.3f}")
print(f"{'Macro F1':<25} {sym_mac:>12.3f} {bert_cross_mac_f1:>12.3f}")
print(f"{'Weighted F1':<25} {sym_wei:>12.3f} {bert_cross_wei_f1:>12.3f}")
print("="*60)

In [ ]:
# Visualisation
metrics = ['Accuracy', 'Macro F1', 'Weighted F1']
sym_scores  = [sym_acc,          sym_mac,           sym_wei]
bert_scores = [bert_cross_acc,   bert_cross_mac_f1, bert_cross_wei_f1]

x, width = np.arange(len(metrics)), 0.35
fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - width/2, sym_scores,  width, label='Symbolique (FrameNet)', color='#e07b54')
bars2 = ax.bar(x + width/2, bert_scores, width, label='BERT',                  color='#4c8bb5')

for bar in bars1 + bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.2f}', ha='center', fontsize=10)

ax.set_xticks(x); ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.15); ax.set_ylabel('Score')
ax.set_title('Symbolique vs BERT — Dataset FrameNet Roles')
ax.legend()
plt.tight_layout()
plt.savefig('comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Inference — Test sur de nouveaux exemples

Même principe que dans le notebook `token_classification` : on utilise un `pipeline` HuggingFace.

In [ ]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
)

examples = [
    "She fell into a deep depression.",          # CONTAINMENT
    "The project is moving forward steadily.",   # SOURCE_PATH_GOAL
    "Prices have risen sharply this year.",      # VERTICALITY
    "He pushed through all obstacles.",          # FORCE
]

print("=== Inférence — Nouveaux exemples ===")
for text in examples:
    result = classifier(text)[0]
    print(f"  '{text}'")
    print(f"  → {result['label']}  (score : {result['score']:.3f})\n")

---
## 8. Explicabilité — LIME

In [ ]:
import torch
from lime import lime_text
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def predict_for_lime(sentences):
    """Adapte le modèle HuggingFace pour LIME."""
    model.eval()
    encoded = tokenizer(
        sentences, truncation=True, padding=True, max_length=128, return_tensors='pt'
    ).to(device)
    with torch.no_grad():
        logits = model(**encoded).logits
    probs = torch.softmax(logits, dim=1).cpu().numpy()
    return probs

explainer = lime_text.LimeTextExplainer(class_names=LABELS, verbose=False)

In [ ]:
# LIME sur 3 exemples représentatifs
for cls in ['CONTAINMENT', 'SOURCE_PATH_GOAL', 'VERTICALITY']:
    cls_id   = label2id[cls]
    examples_cls = test_df[test_df['label_id'] == cls_id].head(1)
    for _, row in examples_cls.iterrows():
        text = str(row['LinguisticExamples'])
        print(f"\n--- {cls} ---\nTexte : {text}")
        exp = explainer.explain_instance(
            text, predict_for_lime, num_features=6, top_labels=3, num_samples=100
        )
        exp.show_in_notebook()

In [ ]:
# Explication globale LIME — mots les plus influents par classe
global_importance = [{} for _ in LABELS]

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="LIME global"):
    text = str(row['LinguisticExamples'])
    exp  = explainer.explain_instance(
        text, predict_for_lime, num_features=20, top_labels=NUM_LABELS, num_samples=100
    )
    pred_idx = exp.top_labels[0]
    for word, importance in exp.as_list(label=pred_idx):
        global_importance[pred_idx].setdefault(word, []).append(importance)

global_avg = [{w: np.mean(v) for w, v in d.items()} for d in global_importance]

# Visualisation
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
for i, ax in enumerate(axes.flatten()[:NUM_LABELS]):
    top = sorted(global_avg[i].items(), key=lambda x: x[1], reverse=True)[:10]
    top.reverse()
    if top:
        ax.barh(range(len(top)), [v for _, v in top], color='steelblue')
        ax.set_yticks(range(len(top)))
        ax.set_yticklabels([w for w, _ in top])
    ax.set_title(LABELS[i], fontsize=10)

plt.suptitle('Mots influents par schéma — LIME global', fontsize=13)
plt.tight_layout()
plt.savefig('lime_global.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 9. Synthèse

| Critère | Symbolique (FrameNet) | BERT (bert-base-uncased) |
|---|---|---|
| **Principe** | Règles FrameNet + ontologie | Transformer pré-entraîné fine-tuné |
| **Accuracy exacte** | ~21% | ~75% |
| **Weighted F1** | ~0.40 | ~0.80 |
| **Interprétabilité** | Intrinsèque | Post-hoc (LIME) |
| **Données nécessaires** | Aucune | Exemples annotés |

**Conclusion :** BERT surpasse largement l'approche symbolique. Le `Trainer` HuggingFace simplifie l'entraînement tout en restant configurable. LIME permet d'interpréter les décisions du modèle.